In [1]:
import pandas as pd
from ucimlrepo import fetch_ucirepo 

In [2]:
RAW_OUTPUT_FILE = r"../data/drug_consumption_raw.csv"
TRANSFORMED_OUTPUT_FILE = r"../data/drug_consumption_transformed.csv"

In [3]:
# fetch dataset 
drug_consumption_quantified = fetch_ucirepo(id=373) 
  
# data (as pandas dataframes) 
X = drug_consumption_quantified.data.features 
y = drug_consumption_quantified.data.targets 
  
df = pd.concat([X, y], axis = 1)
df = df.rename(columns = {"impuslive": "impulsive"}) # fix spelling mistake
df.head()

,age,gender,education,country,ethnicity,nscore,escore,oscore,ascore,cscore,...,ecstasy,heroin,ketamine,legalh,lsd,meth,mushrooms,nicotine,semer,vsa
0,0.49788,0.48246,-0.05921,0.96082,0.12600,0.31287,-0.57545,-0.58331,-0.91699,-0.00665,...,CL0,CL0,CL0,CL0,CL0,CL0,CL0,CL2,CL0,CL0
1,-0.07854,-0.48246,1.98437,0.96082,-0.31685,-0.67825,1.93886,1.43533,0.76096,-0.14277,...,CL4,CL0,CL2,CL0,CL2,CL3,CL0,CL4,CL0,CL0
2,0.49788,-0.48246,-0.05921,0.96082,-0.31685,-0.46725,0.80523,-0.84732,-1.62090,-1.01450,...,CL0,CL0,CL0,CL0,CL0,CL0,CL1,CL0,CL0,CL0
3,-0.95197,0.48246,1.16365,0.96082,-0.31685,-0.14882,-0.80615,-0.01928,0.59042,0.58489,...,CL0,CL0,CL2,CL0,CL0,CL0,CL0,CL2,CL0,CL0
4,0.49788,0.48246,1.98437,0.96082,-0.31685,0.73545,-1.63340,-0.45174,-0.30172,1.30612,...,CL1,CL0,CL0,CL1,CL0,CL0,CL2,CL2,CL0,CL0


In [4]:
df.to_csv(RAW_OUTPUT_FILE, index = False)

In [5]:
# df = pd.read_csv("../data/drug_consumption_transformed.csv")

In [6]:
# Ordered from lowest to highest numerical value (I think, I did this manually)
COLUMN_CATEGORIES = {   
    "age": ["18-24", "25-34", "35-44", "45-54", "55-64", "65+"],
    "gender": ["Male", "Female"],
    "education": ["Left school before 16 years", "Left school at 16 years", "Left school at 17 years", "Left school at 18 years",
                "Some college or university, no certificate or degree", "Professional certificate/diploma", "University degree",
                "Masters degree", "Doctorate degree"],
    "country": ["USA", "New Zealand", "Other", "Australia", "Republic of Ireland", "Canada", "UK"],
    "ethnicity": ["Black", "Asian", "White", "Mixed-White/Black", "Other", "Mixed-White/Asian", "Mixed-Black/Asian"],
    "nscore": range(12, 61),
    "escore": [i for i in range(16, 60) if i not in [17, 57]],
    "oscore": [i for i in range(24, 61) if i not in [25, 27]],
    "ascore": [i for i in range(12, 61) if i not in [13, 14, 15, 17, 19, 20, 21, 22]],
    "cscore": [i for i in range(17, 60) if i not in [18, 58]]
}

In [7]:
df2 = df.copy() # to be transformed
for column, categories in COLUMN_CATEGORIES.items():
    assert len(categories) == df2[column].nunique(), column # make sure I included all the categories

    sorted_numbers = sorted(df2[column].unique())

    column_mapping = {number: category for number, category in zip(sorted_numbers, categories)}
    
    df2[column] = df2[column].map(column_mapping)

df2.head()

# combine left school levels
df2["education"] = df2["education"].replace(["Left school before 16 years", "Left school at 16 years", "Left school at 17 years", "Left school at 18 years"], "Left school")

# only keep chocolate response
df2 = df2[[*df2.columns[:10], "choc"]]

In [9]:
df2

,age,gender,education,country,ethnicity,nscore,escore,oscore,ascore,cscore,choc
0,35-44,Female,Professional certificate/diploma,UK,Mixed-White/Asian,39,36,42,37,42,CL5
1,25-34,Male,Doctorate degree,UK,White,29,52,55,48,41,CL6
2,35-44,Male,Professional certificate/diploma,UK,White,31,45,40,32,34,CL4
3,18-24,Female,Masters degree,UK,White,34,34,46,47,46,CL4
4,35-44,Female,Doctorate degree,UK,White,43,28,43,41,50,CL6
...,...,...,...,...,...,...,...,...,...,...,...
1880,18-24,Female,"Some college or university, no certificate or ...",USA,White,25,51,57,48,33,CL4
1881,18-24,Male,"Some college or university, no certificate or ...",USA,White,33,51,50,48,30,CL4
1882,25-34,Female,University degree,USA,White,47,30,37,31,31,CL6
1883,18-24,Female,"Some college or university, no certificate or ...",USA,White,45,26,48,32,22,CL5


In [8]:
df2.to_csv(TRANSFORMED_OUTPUT_FILE, index = False)